# Read Data and Download Libraries

In [ ]:
!pip install --upgrade transformers torch datasets torchvision --quiet
!pip install datasets seqeval evaluate huggingface_hub matplotlib onnxruntime seaborn huggingface-hub --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.2 MB/s eta 0:00:00


In [ ]:
# Same code for feeding the data as in 'Retrain_Models' to make sure that the model sees the same style of data as it was trained on
def open_file_get_data_bios(filepath):
    words = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as file:
        word = []
        label = []
        counter = 0
        for line in file:
            split_lines = line.split()
            if len(split_lines) > 0:
                if counter == 128 or (split_lines[0] == "." and counter > 100):
                    if len(split_lines) != 0:
                        if split_lines[0] == ".":
                            word.append(split_lines[0])
                            label.append(split_lines[-1])
                    if len(word) != 0:
                        words.append(word)
                        labels.append(label)
                    word = []
                    label = []
                    counter = 0
                    continue

                word.append(split_lines[0])
                label.append(split_lines[-1])
                counter += 1
    return words, labels

words, labels = open_file_get_data_bios("test_spacy.txt")

wordsTest = words
labelsTest = labels

# Model Inference on each token and then putting tokens back into words

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import Dataset
import numpy as np

# Load the models as a matrix so that I can run them in a for loop
model_checkpoint = ["pabRomero/BERT-full-finetuned-ner-pablo",
                    "pabRomero/BioBERT-full-finetuned-ner-pablo",
                    "pabRomero/ClinicalBERT-full-finetuned-ner-pablo",
                    "pabRomero/BioClinicalBERT-full-finetuned-ner-pablo",
                    "pabRomero/PubMedBERT-full-finetuned-ner-pablo",
                    "pabRomero/BioMedRoBERTa-full-finetuned-ner-pablo",
                    "pabRomero/RoBERTa-full-finetuned-ner-pablo",
                    "pabRomero/RoBERTa-Large-full-finetuned-ner-pablo"]

# model_checkpoint = ["pabRomero/BioMedRoBERTa-finetuned-valid-testing-0.00005-32"]

label_to_tag = {
    'O':0,
    'B-Drug':1, 'I-Drug':2,
    'B-Reason':3, 'I-Reason':4,
    'B-Route':5, 'I-Route':6,
    'B-Strength':7, 'I-Strength':8,
    'B-Form':9, 'I-Form':10,
    'B-Dosage':11, 'I-Dosage':12,
    'B-Frequency':13, 'I-Frequency':14,
    'B-Duration':15, 'I-Duration':16,
    'B-ADE':17, 'I-ADE':18,
}

id_to_label = {i:v for i, v in enumerate(label_to_tag)}

# This is where the predictions on each word will be stored, the length will be equal to the number of models then the number of samples then the number of words in those samples
ensemble_labels = []
ensemble_logits = []

# This is the same but token level instead
ensemble_token_predictions = []
ensemble_token_labels = []

for model in model_checkpoint:
    # Load the model and the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model, add_prefix_space=True)
    model = AutoModelForTokenClassification.from_pretrained(model)

    # Move model to GPU if available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    # Create a dataset
    datasetTest = Dataset.from_dict({"tokens": wordsTest})

    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        return tokenized_inputs

    # Tokenize the dataset with the .map function
    tokenized_datasetTest = datasetTest.map(tokenize_and_align_labels, batched=True, remove_columns=datasetTest.column_names)

    results = []
    with torch.inference_mode():
        for item in tokenized_datasetTest:
            input_ids = torch.tensor([item['input_ids']]).to(device)
            attention_mask = torch.tensor([item['attention_mask']]).to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Move results back to CPU
            results.append({
                'logits': logits.cpu().numpy(),
                'tokens': tokenizer.convert_ids_to_tokens(item['input_ids'])
            })

    # Here you can choose the method to put the token predictions back into words, so first being 'para' '##ce' '##ta' '##mol' each of the tokens have their own probability, so the first token is the only one that is taken into consideration. Max is the token in the sub word that has the highest raw logit probability, making sure to account for B-, so if the token is I- but the token starts as B- then it is changed to B-. Average is the average of all logits, making sure to account for B- as the start once again, then take the argmax of the average logits.
    maxVoting = False
    firstVoting = True
    averageVoting = False

    resultCounter = 0
    finalLabels = []
    finalLogits = []

    finaltokenPredictions = []
    finaltokenLabels = []
    for i, sentences in enumerate(wordsTest):
        resultCounter = 1
        for y, word in enumerate(sentences):
            tokens = ""
            logits = []
            lowerWord = word.lower()
            # Take the first word in the sequence and add tokens to it until it equals the word, so for paracetamol, it would start with para then ce then ta, etc. Until the word is complete, at the same time, I add in the logits to a matrix to then do the first, average or max to put the word back together.
            while tokens != lowerWord:
                token = results[i]['tokens'][resultCounter].lower()
                if token[:2] == "##":
                    tokens += token[2:]
                elif token[0] == "ġ":
                    tokens += token[1:]
                else:
                    tokens += token
                logits.append(results[i]['logits'][0][resultCounter])
                resultCounter += 1

            if maxVoting:
                firstLabel = np.argmax(logits[0])
                if len(logits) > 1:
                    # If the first token predicts a B- tag (odd number and not 0)
                    if firstLabel != 0 and firstLabel % 2 != 0:
                        for o, logit in enumerate(logits):
                            if o != 0:
                                maxLogit = np.argmax(logit)
                                if maxLogit % 2 == 0 and maxLogit != 0:
                                    logit[maxLogit - 1] = logit[maxLogit]
                                    logit[maxLogit] = 0

                    posOfMaxLogit = np.argmax(np.max(logits, axis=1))
                    label = np.argmax(logits[posOfMaxLogit])
                    logitsOut = logits[posOfMaxLogit]
                else:
                    label = firstLabel
                    logitsOut = logits[0]

            if firstVoting:
                label = np.argmax(logits[0])
                logitsOut = logits[0]

            if averageVoting:
                firstPos = np.argmax(logits[0])
                if firstPos != 0 and firstPos % 2 != 0:
                    for o, logit in enumerate(logits):
                        if o != 0:
                            maxLogit = np.argmax(logit)
                            if maxLogit % 2 == 0:
                                logit[maxLogit - 1] = logit[maxLogit]
                                logit[maxLogit] = 0
                average_logits = np.mean(logits, axis=0)
                label = np.argmax(average_logits)
                logitsOut = average_logits

            finalLabels.append(label)
            finalLogits.append(logitsOut)

            # This is for token prediction, we just take the logits and do argmax on them, we also need to extend the labels like we did for training but for testing here, so that the tokens and labels have the same length
            previous_prediction = "O"
            for p, logit in enumerate(logits):
                finaltokenPredictions.append(id_to_label[np.argmax(logit)])
                finaltokenLabels.append(labelsTest[i][y])
                if p != 0:
                  if finaltokenLabels[-1][0] == "B":
                    finaltokenLabels[-1] = "I" + finaltokenLabels[-1][1:]
                  if finaltokenPredictions[-1][0] == "B":
                    finaltokenPredictions[-1] = "I" + finaltokenPredictions[-1][1:]
                else:
                  if previous_prediction == "O" and finaltokenPredictions[-1] == "I":
                    finaltokenPredictions[-1] = "B" + finaltokenPredictions[-1][1:]
                previous_prediction = finaltokenPredictions[-1]


    labeledOutput = [id_to_label[output] for output in finalLabels]

    ensemble_labels.append(labeledOutput)
    ensemble_logits.append(finalLogits)

    ensemble_token_predictions.append(finaltokenPredictions)
    ensemble_token_labels.append(finaltokenLabels)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Map:   0%|          | 0/4843 [00:00<?, ? examples/s]

In [ ]:
import onnxruntime as ort
import numpy as np
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

label_to_tag = {
    'O':0,
    'B-Drug':1, 'I-Drug':2,
    'B-Reason':3, 'I-Reason':4,
    'B-Route':5, 'I-Route':6,
    'B-Strength':7, 'I-Strength':8,
    'B-Form':9, 'I-Form':10,
    'B-Dosage':11, 'I-Dosage':12,
    'B-Frequency':13, 'I-Frequency':14,
    'B-Duration':15, 'I-Duration':16,
    'B-ADE':17, 'I-ADE':18,
}

id_to_label = {i:v for i, v in enumerate(label_to_tag)}

def download_onnx_model():
    model_path = hf_hub_download(
        repo_id="pabRomero/BioMedRoBERTa-full-finetuned-ner-pablo",
        filename="onnx/model_quantized.onnx"
    )
    return model_path

def evaluate_onnx_model(test_file, model_path):
    print("Initializing CPU evaluation...")

    tokenizer = AutoTokenizer.from_pretrained(
        "pabRomero/BioMedRoBERTa-full-finetuned-ner-pablo",
        add_prefix_space=True
    )

    # Basic session optimization for CPU
    session_options = ort.SessionOptions()
    session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    session_options.intra_op_num_threads = 4  # Adjust based on CPU cores

    # Simple CPU session creation
    session = ort.InferenceSession(model_path, sess_options=session_options)
    print("Model loaded successfully on CPU")

    words, labels = open_file_get_data_bios(test_file)
    words = words
    labels = labels
    total_sentences = len(words)
    print(f"Processing {total_sentences} sentences...")

    finalLabels = []
    finalLogits = []
    finaltokenPredictions = []
    finaltokenLabels = []

    # Process one sentence at a time
    for i, sentence in enumerate(words):
        if i % 50 == 0:  # More frequent progress updates
            print(f"Processing sentence {i}/{total_sentences} ({(i/total_sentences*100):.1f}%)")

        inputs = tokenizer(
            sentence,
            return_tensors="pt",
            is_split_into_words=True,
            truncation=True
        )

        ort_inputs = {
            'input_ids': inputs.input_ids.numpy(),
            'attention_mask': inputs.attention_mask.numpy()
        }

        logits = session.run(None, ort_inputs)[0][0]

        for y, word in enumerate(sentence):
            word_tokens = tokenizer.encode(word, add_special_tokens=False)
            word_start_idx = inputs.word_to_tokens(y).start

            label = np.argmax(logits[word_start_idx])
            finalLabels.append(label)
            finalLogits.append(logits[word_start_idx])

            previous_prediction = "O"
            for p, token_idx in enumerate(range(word_start_idx, word_start_idx + len(word_tokens))):
                token_label = id_to_label[np.argmax(logits[token_idx])]
                finaltokenPredictions.append(token_label)
                finaltokenLabels.append(labels[i][y])

                if p != 0:
                    if finaltokenLabels[-1][0] == "B":
                        finaltokenLabels[-1] = "I" + finaltokenLabels[-1][1:]
                    if finaltokenPredictions[-1][0] == "B":
                        finaltokenPredictions[-1] = "I" + finaltokenPredictions[-1][1:]
                else:
                    if previous_prediction == "O" and finaltokenPredictions[-1][0] == "I":
                        finaltokenPredictions[-1] = "B" + finaltokenPredictions[-1][1:]
                previous_prediction = finaltokenPredictions[-1]

    print("Processing complete!")
    labeledOutput = [id_to_label[output] for output in finalLabels]
    return [labeledOutput], [finalLogits], [finaltokenPredictions], [finaltokenLabels]

# Run evaluation
model_path = download_onnx_model()
ensemble_labels, ensemble_logits, ensemble_token_predictions, ensemble_token_labels = evaluate_onnx_model("test_spacy.txt", model_path)

Initializing CPU evaluation...
Model loaded successfully on CPU
Processing 4843 sentences...
Processing sentence 0/4843 (0.0%)
Processing sentence 50/4843 (1.0%)
Processing sentence 100/4843 (2.1%)
Processing sentence 150/4843 (3.1%)
Processing sentence 200/4843 (4.1%)
Processing sentence 250/4843 (5.2%)
Processing sentence 300/4843 (6.2%)
Processing sentence 350/4843 (7.2%)
Processing sentence 400/4843 (8.3%)
Processing sentence 450/4843 (9.3%)
Processing sentence 500/4843 (10.3%)
Processing sentence 550/4843 (11.4%)
Processing sentence 600/4843 (12.4%)
Processing sentence 650/4843 (13.4%)
Processing sentence 700/4843 (14.5%)
Processing sentence 750/4843 (15.5%)
Processing sentence 800/4843 (16.5%)
Processing sentence 850/4843 (17.6%)
Processing sentence 900/4843 (18.6%)
Processing sentence 950/4843 (19.6%)
Processing sentence 1000/4843 (20.6%)
Processing sentence 1050/4843 (21.7%)
Processing sentence 1100/4843 (22.7%)
Processing sentence 1150/4843 (23.7%)
Processing sentence 1200/484

In [ ]:
# ensemble_labels = [ensemble_labels]
# ensemble_logits = [ensemble_logits]
# ensemble_token_predictions = [ensemble_token_predictions]
# ensemble_token_labels = [ensemble_token_labels]

In [ ]:
len(ensemble_labels[0])

563329

# Model Evaluation and Saving

In [ ]:
saveModelResults = True

In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd
from matplotlib.colors import LogNorm
import matplotlib as mpl
import numpy as np

# Function to create a new folder if it doesn't exist
def create_folder(folder_name):
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    return folder_name

# Function to save a confusion matrix as a heatmap
def save_confusion_matrix(y_true, y_pred, labels, folder, filename, model_name):
    # Create confusion matrix

    # labels = ['O', 'Drug', 'Reason', 'Route', 'Strength', 'Form', 'Dosage', 'Frequency', 'Duration', 'ADE']

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Convert to DataFrame for easier plotting
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)

    # Create figure and axes
    fig, ax = plt.subplots(figsize=(14, 9))

    # Plot heatmap with logarithmic color scale
    sns.heatmap(df_cm, annot=True, cmap=plt.cm.Oranges, fmt='.0f', norm=LogNorm())

    # Set title and labels
    plt.title(f"{model_name} - Confusion Matrix on the Test Dataset N2C2 2018\n", fontsize=19)
    plt.xlabel('\nPredicted Labels', fontsize=15)
    plt.ylabel('True Labels\n', fontsize=15)

    # Adjust layout and save figure
    plt.tight_layout(pad=1.1)
    plt.savefig(os.path.join(folder, f'{filename}.png'), dpi=200, bbox_inches='tight')
    plt.close()

# Function to save classification report as a heatmap
def save_classification_report(report, folder, filename, model_name):
    # Convert report to DataFrame
    df = pd.DataFrame(report).transpose()

    rows, cols = df.shape

    # Create mask for all columns except the last (support)
    mask = np.zeros(df.shape)
    mask[:,cols-1] = True

    fig, ax = plt.subplots(figsize=(9, 9))

    # Plot heatmap for precision, recall, and f1-score
    ax = sns.heatmap(df, mask=mask, annot=True, cmap=plt.cm.Purples, fmt='.3g',
            vmin=0.0, vmax=1.0,
            linewidths=1, linecolor='black')

    # Create mask for all columns except the last (support)
    mask = np.zeros(df.shape)
    mask[:,:cols-1] = True

    # Plot heatmap for support column with different color normalization
    ax = sns.heatmap(df, mask=mask, annot=True, cmap=plt.cm.Purples, cbar=False,
            linewidths=1, linecolor='black', fmt='.0f',
            vmin=df['support'].min(),
            vmax=df['support'].sum(),
            norm=mpl.colors.Normalize(vmin=df['support'].min(),
                                      vmax=df['support'].sum()))

    # Set title and adjust labels
    plt.title(f"{model_name} - Classification Report on the Test Dataset N2C2 2018\n", fontsize=14)
    plt.xticks(rotation = 45)
    plt.yticks(rotation = 0)

    # Adjust layout and save figure
    plt.tight_layout(pad=1.1)
    plt.savefig(os.path.join(folder, f'{filename}.png'), dpi=200, bbox_inches='tight')
    plt.close()

# List of model names used in the experiment
# model_names = ["BERT",
#                "BioBERT",
#                "ClinicalBERT",
#                "BioClinicalBERT",
#                "PubMedBERT",
#                "BioMedRoBERTa",
#                "RoBERTa",
#                "RoBERTa-Large"]

model_names = ["BioMedRoBERTa - Quantized (4 bit)"]

## Single Model Evaluation

In [ ]:
len(ensemble_logits[0])

563329

In [ ]:
notBios = False

def not_bios_matrix(matrix):
    for i, modelPred in enumerate(matrix):
        for j, pred in enumerate(modelPred):
            if pred.startswith('B-') or pred.startswith('I-'):
                matrix[i][j] = pred[2:]
    return matrix

if notBios:
    ensemble_labels = not_bios_matrix(ensemble_labels)
    labelsTest = not_bios_matrix(labelsTest)
    ensemble_token_predictions = not_bios_matrix(ensemble_token_predictions)
    ensemble_token_labels = not_bios_matrix(ensemble_token_labels)

In [ ]:
flattenedLabels = [item for sublist in labelsTest for item in sublist]
len(flattenedLabels)

563329

In [ ]:
# This section is responsible for saving and analyzing the results of multiple NER models

# Check if we should save the model results
if saveModelResults:
    # Create a main folder to store all results
    main_folder = create_folder('/content/Ensemble Model Results')

    # Iterate through each model
    for i in range(len(model_names)):
        # Create a folder for the current model
        model_folder = create_folder(os.path.join(main_folder, model_names[i]))

        # Create a subfolder for word-level analysis
        model_folder_word = create_folder(os.path.join(model_folder, "Word Level"))

        print(model_names[i])

        # Generate classification report for word-level predictions
        reportTable = classification_report(flattenedLabels, ensemble_labels[i], digits=4)
        print(reportTable)
        reportDict = classification_report(flattenedLabels, ensemble_labels[i], output_dict=True)

        # Save word-level results to a text file
        with open(f"{model_folder_word}/{model_names[i]}_word_bio_text.txt", 'w') as f:
            f.write(reportTable + "\n")
            f.write(str(reportDict))

        # Save classification report and confusion matrix for word-level analysis
        save_classification_report(reportDict, model_folder_word, f"{model_names[i]}_word_bios", model_names[i])
        save_confusion_matrix(flattenedLabels, ensemble_labels[i], list(id_to_label.values()), model_folder_word, f"{model_names[i]}_word_confusion_matrix", model_names[i])

        # Create a subfolder for token-level analysis
        model_folder_token = create_folder(os.path.join(model_folder, "Token Level"))

        # Generate classification report for token-level predictions
        reportTable = classification_report(ensemble_token_labels[i], ensemble_token_predictions[i], digits=4)
        print(reportTable)
        reportDict = classification_report(ensemble_token_labels[i], ensemble_token_predictions[i], output_dict=True)

        # Save token-level results to a text file
        with open(f"{model_folder_token}/{model_names[i]}_token_bio_text.txt", 'w') as f:
            f.write(reportTable + "\n")
            f.write(str(reportDict))

        # Save classification report and confusion matrix for token-level analysis
        save_classification_report(reportDict, model_folder_token, f"{model_names[i]}_token_bios", model_names[i])
        save_confusion_matrix(ensemble_token_labels[i], ensemble_token_predictions[i], list(id_to_label.values()), model_folder_token, f"{model_names[i]}_token_confusion_matrix", model_names[i])

        print("\n")

BioMedRoBERTa - Quantized (4 bit)
              precision    recall  f1-score   support

       B-ADE     0.5588    0.6094    0.5830       663
    B-Dosage     0.9349    0.8913    0.9126      2869
      B-Drug     0.9355    0.9321    0.9338     10916
  B-Duration     0.8058    0.7488    0.7762       410
      B-Form     0.9367    0.9324    0.9346      4558
 B-Frequency     0.8987    0.8272    0.8615      4989
    B-Reason     0.6336    0.6582    0.6457      2724
     B-Route     0.9601    0.9460    0.9530      3534
  B-Strength     0.9661    0.9674    0.9667      4327
       I-ADE     0.4617    0.4597    0.4607       459
    I-Dosage     0.9403    0.9699    0.9549      5519
      I-Drug     0.7125    0.7965    0.7522      2029
  I-Duration     0.7710    0.8431    0.8054       599
      I-Form     0.8717    0.9196    0.8950      2327
 I-Frequency     0.7322    0.9153    0.8136      7176
    I-Reason     0.5239    0.5100    0.5168      2002
     I-Route     0.8186    0.7126    0.7619    

In [ ]:
flattenedLabels = [item for sublist in labelsTest for item in sublist]
len(flattenedLabels)

563329

## Ensemble Model Evaluation

In [ ]:
import random
import numpy as np

# Transpose the ensemble_labels to group predictions for each token
# This allows us to consider all model predictions for a single token at once
transposed_labels = np.array(ensemble_labels).T

# Flags to determine which voting method to use
majority_voting = True  # Requires at least 4 models to agree
most_voted = False        # Selects the label with the most votes
most_voted_random = False  # Randomly selects among labels with equal (highest) votes

voted_labels = []  # This will store the final voted labels for each token

# Iterate through each token's set of predictions
for voting_labels in transposed_labels:
    # Count the occurrences of each unique label for this token
    unique, counts = np.unique(voting_labels, return_counts=True)

    if majority_voting:
        # Majority voting: label must have at least 4 votes
        if np.max(counts) >= 4:
            max_index = np.argmax(counts)
            voted_labels.append(unique[max_index])
        else:
            voted_labels.append("O")  # If no majority, label as "Outside" (not an entity)

    if most_voted:
        if len(counts) == 1:
            # If all models agree, use that label
            voted_labels.append(unique[0])
        else:
            # Find the label(s) with the most votes
            max_index = np.argmax(counts)

            if most_voted_random:
                # Handle cases where multiple labels have the same (highest) vote count
                matching_label = []
                for i, count in enumerate(counts):
                    if count == counts[max_index]:
                        matching_label.append(unique[i])

                if len(matching_label) > 1:
                    # Randomly select among the labels with the highest vote count
                    voted_labels.append(matching_label[random.randint(0, len(matching_label) - 1)])
                else:
                    voted_labels.append(unique[max_index])
            else:
                # If not using random selection, just choose the first label with max votes
                voted_labels.append(unique[max_index])

# At this point, voted_labels contains the final ensemble predictions for each token

In [ ]:
# Create folders to store the results
model_folder = create_folder(os.path.join(main_folder, "Ensemble Voting"))
model_folder_word = create_folder(os.path.join(model_folder, "Word Level"))

# Generate classification report
reportTable = classification_report(flattenedLabels, voted_labels, digits=4)
print(reportTable)
reportDict = classification_report(flattenedLabels, voted_labels, output_dict=True)

# Save the classification report as text
with open(f"{model_folder_word}/Ensemble_Voting_word_bio_text.txt", 'w') as f:
    f.write(reportTable + "\n")
    f.write(str(reportDict))

# Save the classification report as a visual heatmap
save_classification_report(reportDict, model_folder_word, f"Ensemble_Voting_word_bios", "Voted Ensemble")

# Save the confusion matrix as a heatmap
save_confusion_matrix(flattenedLabels, voted_labels, list(id_to_label.values()),
                      model_folder_word, f"Ensemble_Voting_word_confusion_matrix", "Voted Ensemble")

## Stacked Ensemble Model Evaluation

In [ ]:
import onnxruntime as ort
import numpy as np

stacked = True
if stacked:
    onnx_model_path = "feedforward_model_stacked.onnx"
    ort_session = ort.InferenceSession(onnx_model_path)
    input_name = ort_session.get_inputs()[0].name

    stacked_labels = []
    for i in range(len(ensemble_logits[0])):
        # Check if at least two models predict a non-O label
        non_o_predictions = sum(np.argmax(ensemble_logits[y][i]) != 0 for y in range(len(ensemble_logits)))
        if non_o_predictions >= 0:
            # Convert logits to one-hot encoded vectors
            one_hot_input = []
            for y in range(len(ensemble_logits)):
                one_hot = np.zeros(19)  # Assuming 19 classes
                one_hot[np.argmax(ensemble_logits[y][i])] = 1
                one_hot_input.extend(one_hot)

            # Prepare input for ONNX model and add the dimension so that it is compatible with .onnx model [1,152]
            onnx_input = np.array(one_hot_input, dtype=np.float32).reshape(1, -1)

            # Run ONNX model inference
            ort_inputs = {input_name: onnx_input}
            ort_outputs = ort_session.run(None, ort_inputs)
            onnx_output = ort_outputs[0]

            # Get the predicted label index
            predicted_label_index = np.argmax(onnx_output)

            # Convert index to text label
            text_output = id_to_label[predicted_label_index]

            stacked_labels.append(text_output)
        else:
            stacked_labels.append("O")

In [ ]:
if stacked:
  model_folder = create_folder(os.path.join(main_folder, "Stacked Ensemble"))
  model_folder_word = create_folder(os.path.join(model_folder, "Word Level"))

  reportTable = classification_report(flattenedLabels, stacked_labels, digits=4)
  print(reportTable)
  reportDict = classification_report(flattenedLabels, stacked_labels, output_dict=True)

  with open(f"{model_folder_word}/Stacked_Ensemble_word_bio_text.txt", 'w') as f:
      f.write(reportTable + "\n")
      f.write(str(reportDict))
  save_classification_report(reportDict, model_folder_word, f"Stacked_Ensemble_word_bios", "Stacked Ensemble")
  save_confusion_matrix(flattenedLabels, stacked_labels, list(id_to_label.values()), model_folder_word, f"Stacked_Ensemble_word_confusion_matrix", "Stacked Ensemble")

              precision    recall  f1-score   support

       B-ADE     0.7222    0.4902    0.5840       663
    B-Dosage     0.9318    0.9094    0.9204      2869
      B-Drug     0.9294    0.9424    0.9358     10916
  B-Duration     0.8543    0.7293    0.7868       410
      B-Form     0.9673    0.9212    0.9437      4558
 B-Frequency     0.9377    0.8054    0.8665      4989
    B-Reason     0.7419    0.6384    0.6863      2724
     B-Route     0.9479    0.9522    0.9500      3534
  B-Strength     0.9712    0.9653    0.9682      4327
       I-ADE     0.5827    0.3377    0.4276       459
    I-Dosage     0.9414    0.9772    0.9589      5519
      I-Drug     0.7395    0.7708    0.7548      2029
  I-Duration     0.7918    0.8381    0.8143       599
      I-Form     0.8655    0.9544    0.9078      2327
 I-Frequency     0.7195    0.9438    0.8165      7176
    I-Reason     0.6053    0.4765    0.5333      2002
     I-Route     0.7957    0.7409    0.7673       247
  I-Strength     0.9576    

In [ ]:
# import numpy as np

# def logit_to_one_hot(logits):
#     one_hot = np.zeros_like(logits)
#     one_hot[np.argmax(logits)] = 1
#     return one_hot

# with open('stacked_data_output.txt', 'w') as file:
#     for i in range(len(ensemble_logits[0])):
#         # Check if at least two models predict a non-O label
#         non_o_predictions = sum(np.argmax(ensemble_logits[y][i]) != 0 for y in range(len(ensemble_logits)))
#         if non_o_predictions >= 2:
#             line = []
#             for y in range(len(ensemble_logits)):
#                 one_hot = logit_to_one_hot(ensemble_logits[y][i])
#                 line.extend([str(int(val)) for val in one_hot])
#             line.append(str(flattenedLabels[i]))
#             file.write(' '.join(line) + '\n')

## Export Data to Google Drive

In [ ]:
from google.colab import drive
import shutil
import os

# You can't directly export the folder we created so we need to copy it to google drive with this code, once the code reaches here it will ask you to log-in, once you do this you will find a folder in your drive
drive.mount('/content/drive')

# Define source and destination paths
src = '/content/Ensemble Model Results'  # Replace with your folder path in Colab
dst = '/content/drive/MyDrive/ONNX 4 Bits Results'  # Replace with desired path in Google Drive

   # Copy the entire folder structure
shutil.copytree(src, dst)

print("Folder structure copied to Google Drive successfully!")

Mounted at /content/drive
Folder structure copied to Google Drive successfully!
